# 갓생살기 인증 사진 분류 모델 — 전이학습 (MobileNetV2)

인증 사진이 챌린지 카테고리(운동 · 공부 · 독서 · 요리 · 기타)와 맞는지 가려내는 모델을 **직접 학습**한다.

- ImageNet 으로 사전학습된 MobileNetV2 를 가져와, **마지막 분류층을 우리 카테고리 5개로 바꿔 우리 사진으로 다시 학습**한다 (전이학습 · 파인튜닝).
- 학습이 끝나면 체크포인트(`godlife-mobilenetv2.pt`)를 내려받아 `ai-server/models/` 에 넣는다. ai-server(FastAPI)가 이 파일을 읽어 추론한다.
- 외부 AI API 는 쓰지 않는다.

**실행 방법**: Colab 메뉴 → 런타임 → 런타임 유형 변경 → **GPU** 선택 후, 위에서부터 차례로 실행한다.

## 1. 데이터 준비

구글 드라이브에 아래처럼 폴더를 만들고 사진을 넣는다. **폴더 이름이 곧 라벨**이고, 서비스의 `categories.ai_label` 과 같아야 한다.

```
MyDrive/godlife-dataset/
├── exercise/   운동 (헬스장, 러닝, 요가, 운동 기구, 운동 중인 모습 …)
├── study/      공부 (노트 필기, 문제집, 노트북 화면, 책상 …)
├── reading/    독서 (펼친 책, 책 표지, 전자책 …)
├── cooking/    요리 (조리 중인 음식, 완성한 요리, 도마 · 냄비 …)
└── other/      기타 생활 습관 (아침 기상 인증, 산책 풍경, 물병 · 물컵 …)
```

- 클래스마다 **150장 이상**을 권한다 (적어도 80장). 클래스끼리 장수가 비슷해야 한쪽으로 쏠리지 않는다.
- 실제 인증 사진과 비슷하게: 휴대폰으로 직접 찍은 사진을 섞고, 밝기 · 각도 · 배경을 다양하게.
- 공개 데이터셋이나 인터넷 사진을 쓸 때는 라이선스를 확인한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/godlife-dataset'   # 사진 폴더
OUT_DIR = '/content/drive/MyDrive/godlife-model'      # 학습 결과를 둘 곳
LABELS = ['exercise', 'study', 'reading', 'cooking', 'other']   # categories.ai_label 과 같은 순서 · 이름
MODEL_VERSION = 'mobilenetv2-v1'

In [ ]:
import os, random
import torch
from torch import nn
from torchvision import datasets, models, transforms

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# 폴더별 사진 수 확인
for label in LABELS:
    folder = os.path.join(DATA_DIR, label)
    count = len([f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]) if os.path.isdir(folder) else 0
    print(f'{label:10s} {count}장' + ('  ← 폴더가 없어요' if not os.path.isdir(folder) else '  ← 80장 이상을 권해요' if count < 80 else ''))

## 2. 전처리와 증강

- 서빙(ai-server/model.py)과 **같은 크기 · 같은 정규화**를 쓴다: 224×224, ImageNet 평균 · 표준편차.
- 학습용에만 증강(잘라내기 · 좌우 반전 · 색 변화 · 회전)을 넣어 적은 사진으로도 덜 외우게 한다.

In [ ]:
IMAGE_SIZE, RESIZE = 224, 256
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(12),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(RESIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# 같은 사진 목록을 두 가지 전처리로 읽고, 클래스마다 8:2 로 나눈다 (검증 사진은 학습에 쓰지 않는다)
full_train = datasets.ImageFolder(DATA_DIR, transform=train_tf)
full_eval = datasets.ImageFolder(DATA_DIR, transform=eval_tf)
assert full_train.classes == sorted(LABELS), f'폴더 이름을 확인해 주세요: {full_train.classes}'
classes = full_train.classes   # ImageFolder 는 폴더 이름을 가나다(알파벳) 순으로 번호 매긴다 → 이 순서를 체크포인트에 저장한다

by_class = {}
for index, (_, target) in enumerate(full_train.samples):
    by_class.setdefault(target, []).append(index)
train_idx, val_idx = [], []
for target, indexes in by_class.items():
    random.shuffle(indexes)
    cut = max(1, int(len(indexes) * 0.2))
    val_idx += indexes[:cut]
    train_idx += indexes[cut:]

train_ds = torch.utils.data.Subset(full_train, train_idx)
val_ds = torch.utils.data.Subset(full_eval, val_idx)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
print('classes:', classes)
print(f'train {len(train_ds)}장 / val {len(val_ds)}장')

## 3. 모델: 사전학습 MobileNetV2 + 새 분류층

- 특징을 뽑는 부분(`features`)은 ImageNet 에서 배운 것을 그대로 쓰고 **얼려 둔다**.
- 마지막 분류층(`classifier[1]`)만 우리 클래스 5개짜리로 바꿔 **새로 학습**한다.

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
for p in model.features.parameters():
    p.requires_grad = False                                   # 사전학습 특징은 얼린다
model.classifier[1] = nn.Linear(model.last_channel, len(classes))   # 1280 → 5
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

def run_epoch(loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum / total, correct / total

## 4. 학습

1. **1단계**: 새 분류층만 학습 (빠르게 자리 잡기)
2. **2단계**: 특징 추출부의 뒤쪽 블록도 풀어서 낮은 학습률로 미세 조정 (파인튜닝)

검증 정확도가 가장 좋았던 가중치를 남긴다.

In [ ]:
import copy
best_acc, best_state, history = 0.0, None, []

def fit(epochs, optimizer, stage):
    global best_acc, best_state
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(train_dl, optimizer)
        va_loss, va_acc = run_epoch(val_dl)
        history.append((stage, epoch, tr_loss, tr_acc, va_loss, va_acc))
        mark = ''
        if va_acc > best_acc:
            best_acc, best_state, mark = va_acc, copy.deepcopy(model.state_dict()), '  ← best'
        print(f'[{stage}] {epoch:2d}  train loss {tr_loss:.3f} acc {tr_acc:.3f} | val loss {va_loss:.3f} acc {va_acc:.3f}{mark}')

# 1단계: 분류층만
fit(8, torch.optim.Adam(model.classifier.parameters(), lr=1e-3), 'head')

# 2단계: 뒤쪽 블록 4개를 풀고 낮은 학습률로
for block in model.features[-4:]:
    for p in block.parameters():
        p.requires_grad = True
fit(6, torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4), 'finetune')

model.load_state_dict(best_state)
print(f'가장 좋은 검증 정확도: {best_acc:.3f}')

## 5. 평가

검증 사진으로 클래스별 정확도와 혼동 행렬을 본다. 어떤 클래스끼리 헷갈리는지 보고, 그 클래스 사진을 더 모으면 된다.

In [ ]:
import numpy as np
model.eval()
confusion = np.zeros((len(classes), len(classes)), dtype=int)
with torch.inference_mode():
    for x, y in val_dl:
        pred = model(x.to(device)).argmax(1).cpu()
        for t, p in zip(y, pred):
            confusion[t, p] += 1

print('행 = 실제, 열 = 예측')
print(' ' * 10 + ''.join(f'{c[:8]:>9s}' for c in classes))
for i, c in enumerate(classes):
    print(f'{c:10s}' + ''.join(f'{n:9d}' for n in confusion[i]) + f'   정확도 {confusion[i, i] / max(1, confusion[i].sum()):.2f}')

## 6. 저장

체크포인트에는 **가중치 · 라벨 순서 · 버전 · 검증 정확도**만 담는다 (ai-server 가 `weights_only=True` 로 안전하게 읽는다).
저장된 `godlife-mobilenetv2.pt` 를 내려받아 프로젝트의 `ai-server/models/` 폴더에 넣으면 된다.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
path = os.path.join(OUT_DIR, 'godlife-mobilenetv2.pt')
torch.save({
    'state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
    'labels': classes,
    'version': MODEL_VERSION,
    'val_accuracy': round(best_acc, 4),
}, path)
print('저장:', path, f'({os.path.getsize(path) / 1e6:.1f} MB)')

from google.colab import files
files.download(path)

## 7. 한 장 시험해 보기

서빙과 같은 전처리로 사진 한 장을 넣어 본다.

In [ ]:
from PIL import Image, ImageOps
from google.colab import files

uploaded = files.upload()
for name in uploaded:
    image = ImageOps.exif_transpose(Image.open(name)).convert('RGB')
    with torch.inference_mode():
        probs = torch.softmax(model(eval_tf(image).unsqueeze(0).to(device)), dim=1)[0].cpu()
    print(name)
    for label, p in sorted(zip(classes, probs.tolist()), key=lambda t: -t[1]):
        print(f'  {label:10s} {p:.3f}')